# Notebook 8: extensions on gene-expression data and five new medical tabular datasets

This notebook adds the experiments a reviewer is most likely to ask for. It uses the same code, budgets, fitness and seeds as Notebook 7.

**New datasets:** five medical tabular datasets from the UCI repository, downloaded from the PMLB mirror on GitHub.

| Dataset | Samples | Features | Classes | Task |
|---|---|---|---|---|
| WDBC | 569 | 30 | 2 | breast tumour, benign vs malignant |
| Hepatitis | 155 | 19 | 2 | survival |
| Dermatology | 366 | 34 | 6 | erythemato-squamous disease type |
| SPECTF | 349 | 44 | 2 | cardiac SPECT, normal vs abnormal |
| Backache | 180 | 32 | 2 | back pain outcome |

**Protocol for the new datasets** (the same as the gene data):
- For each seed, a stratified 70/30 split; min-max scaling fitted on the training part.
- Fitness = 0.99 × (1 − 5-fold CV balanced accuracy of KNN, k = 5) + 0.01 × (fraction of features kept). Balanced accuracy is used because three of the five datasets are imbalanced.
- Test: balanced accuracy, macro-F1 and so on, on the held-out 30%.
- Population 20, budget 1,020 evaluations, ten seeds. Dense start, since *D* < 100.

**What runs where**

| Experiment | Gene data (5) | New tabular (5) |
|---|---|---|
| Main benchmark (7 wrappers + 4 filters) | reused from Notebook 7 | **new** |
| GA+SI and HO+SI | **new** | **new** |
| Q-learning controller, same budget | **new** | **new** |
| κ = 1, 3, 4 | **new** | **new** |
| Random arm / PERTURB only / no intervention | reused from Notebook 7 | **new** |
| 3× budget: UCB1, random arm | reused from Notebook 7 | **new** |
| 3× budget: Q-learning | **new** | **new** |

**Runtime:** about 3–6 hours on Colab in total, split roughly evenly between the gene data and the new tabular datasets. Every run is saved as soon as it finishes, so rerunning a cell resumes where it stopped.

### Fixed plan for the extension experiments (written before any of these runs; every outcome is reported)

**Why these runs:** they answer the main objections a reviewer would raise about the v2 results.
1. *"Is the gain from the controller or from the interventions?"* A second learning controller (tabular **Q-learning**) is added next to UCB1 and random arm choice.
2. *"Would the interventions help any optimiser?"* The same intervention layer (stagnation trigger, PERTURB, RESTART, tabu memory, tie-break, UCB1 with the gain reward) is wrapped around **GA** and **HO**. These are called **GA+SI** and **HO+SI** (SI = stagnation-triggered interventions). Their arms are PERTURB, RESTART and one iteration of the base optimiser.
3. *"How sensitive is RL-DOHO to κ?"* κ ∈ {1, 2, 3, 4}; κ = 2 is the main setting and is reused.
4. *"What does it cost?"* Wall-clock time is split into the algorithm's own overhead and the time spent in the fitness function. The number of distinct subsets each run evaluates is also recorded.

**Q-learning controller (settings fixed now, not tuned)**
- State = budget phase (first, middle or last third of the budget) × whether the previous intervention improved the best subset. That gives 6 states.
- Actions = the four arms. Reward = the same gain reward as UCB1.
- One-step Q-learning: learning rate 0.3, discount 0.5, ε-greedy with ε = 0.2, Q starts at 0.
- The Q-table starts empty in every run, like UCB1.

**Hypotheses (fixed in advance)**
- **H5:** At 3× the budget, Q-learning vs random arm choice. No direction predicted; pooled two-sided sign test on fitness.
- **H6:** GA+SI reaches lower fitness than GA, and HO+SI lower than HO. Pooled two-sided sign tests, Holm-corrected over the two comparisons.
- **H7:** κ has no effect on RL-DOHO's fitness. Friedman test over κ ∈ {1, 2, 3, 4} on the pooled runs, with per-run ranks reported.
- All other comparisons (RL-DOHO against every method on the new datasets, Q-learning vs UCB1 at the same budget, GA+SI vs RL-DOHO) are reported with the same tests as Notebooks 6–7. No direction is predicted for them.

A difference below 1e-12 counts as a tie, as before.

In [1]:
# ================= 1. SETUP =================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os, time, pickle, warnings, urllib.request, gzip, math, hashlib
import numpy as np, pandas as pd, scipy.io as sio, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, precision_score, recall_score, accuracy_score
warnings.filterwarnings("ignore")

Mounted at /content/drive


In [2]:
# ================= 2. CONFIG =================
ROOT      = '/content/drive/MyDrive' if IN_COLAB else '.'
OUT_DIR   = f"{ROOT}/rl_doho_results_v3_knn"                    # results of this notebook
V2_DIR    = f"{ROOT}/rl_doho_results_v2_genes"                  # Notebook 7: runs, ablation, long budget, caches
PREV_DIRS = [f"{ROOT}/rl_doho_results_hd", f"{ROOT}/rl_doho_results_ra", f"{ROOT}/rl_doho_results_baselines_hd", V2_DIR]
NB5_FILE  = f"{ROOT}/rl_doho_results_baselines_hd/baselines_hd_runs_p20_i50.pkl"   # filters on gene data
DATA_DIR  = f"{OUT_DIR}/data"
GENE      = ["colon", "ALLAML", "lung", "GLIOMA", "GSE93272"]
TAB       = {"WDBC": "breast_cancer_wisconsin_diagnostic", "Hepatitis": "hepatitis", "Dermatology": "dermatology",
             "SPECTF": "spectf", "Backache": "backache"}
TABS      = list(TAB)
DATASETS  = GENE + TABS
SEEDS     = list(range(10))
POP_SIZE  = 20
BUDGET    = POP_SIZE * 51                     # 1,020 evaluations, as in Notebooks 2-7
LONG_BUDGET = 3 * BUDGET
KNN_K, CV_FOLDS, TEST_SIZE = 5, 5, 0.3
ALPHA, BETA = 0.99, 0.01
N_TOP_RA  = 5000
SCORING   = {d: "accuracy" for d in GENE[:4]} | {"GSE93272": "balanced_accuracy"} | {d: "balanced_accuracy" for d in TABS}
VERBOSE   = False
RUN = dict(tab_main=True, addons=True, variants=True, long=True, overhead=True)

FAST = False                                  # True: 2 small datasets x 2 seeds, to test the notebook quickly
if FAST: GENE, TABS, SEEDS = ["colon"], ["WDBC"], [0, 1]; DATASETS = GENE + TABS

os.makedirs(DATA_DIR, exist_ok=True)
FAMILY   = ["RL-DOHO", "DO", "HO", "DOHO"]
WRAPPERS = FAMILY + ["GA", "BPSO", "BGWO"]
FILTERS  = ["LASSO", "mRMR", "ReliefF", "All features"]
METHODS  = WRAPPERS + FILTERS
ADDONS   = ["GA+SI", "HO+SI"]
def init_for(D): return "sparse" if D > 100 else "dense"
print("Output folder:", OUT_DIR, "| budget:", BUDGET, "| datasets:", DATASETS)

Output folder: /content/drive/MyDrive/rl_doho_results_v3_knn | budget: 1020 | datasets: ['colon', 'ALLAML', 'lung', 'GLIOMA', 'GSE93272', 'WDBC', 'Hepatitis', 'Dermatology', 'SPECTF', 'Backache']


## 3. Data: the five gene-expression sets (as in Notebook 7) and the five new medical tabular sets

In [3]:
DATA = {}
URL = "https://raw.githubusercontent.com/jundongl/scikit-feature/master/skfeature/data/{}.mat"
def first_existing(paths, default):
    return next((p for p in paths if os.path.exists(p)), default)
for name in [d for d in GENE if d != "GSE93272"]:
    path = first_existing([f"{d}/data/{name}.mat" for d in PREV_DIRS + [V2_DIR]], f"{DATA_DIR}/{name}.mat")
    if not os.path.exists(path): urllib.request.urlretrieve(URL.format(name), path)
    d = sio.loadmat(path); DATA[name] = (d["X"].astype(float), d["Y"].ravel())

if "GSE93272" in GENE:
    gpath = first_existing([f"{d}/data/GSE93272_series_matrix.txt.gz" for d in PREV_DIRS + [V2_DIR]], f"{DATA_DIR}/GSE93272_series_matrix.txt.gz")
    if not os.path.exists(gpath):
        urllib.request.urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE93nnn/GSE93272/matrix/GSE93272_series_matrix.txt.gz", gpath)
    chars = []
    with gzip.open(gpath, "rt") as f:
        for line in f:
            if line.startswith("!Sample_geo_accession"): gsm = [v.strip('"') for v in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics_ch1"): chars.append([v.strip('"') for v in line.rstrip("\n").split("\t")[1:]])
            elif line.startswith("!series_matrix_table_begin"): break
        expr = pd.read_csv(f, sep="\t", index_col=0, comment="!")
    meta = pd.DataFrame(index=gsm)
    for row in chars:
        keys = {v.split(":", 1)[0].strip() for v in row if ":" in v}
        if len(keys) == 1:
            k = keys.pop(); meta[k] = [v.split(":", 1)[1].strip() if ":" in v else None for v in row]
    meta["individual id"] = meta.get("individual id", pd.Series(meta.index, index=meta.index)).fillna(pd.Series(meta.index, index=meta.index))
    meta_k = meta[~meta["individual id"].duplicated(keep="first")]
    DATA["GSE93272"] = (expr[gsm][meta_k.index].T.values.astype(float), (meta_k["disease state"].str.upper() == "RA").astype(int).values)

# New tabular datasets (PMLB copies of the UCI files; checksums printed so the exact files are on record)
PMLB = "https://media.githubusercontent.com/media/EpistasisLab/pmlb/master/datasets/{0}/{0}.tsv.gz"
SHA = {"breast_cancer_wisconsin_diagnostic": "b3845ddf3afff808cd6449daa546caf4d8b18f109b7ebb2344990e4c37fea291",
       "hepatitis": "87df1bbfb83b8204093ccd84ed18bb3695220c03aa4096f63a84cc5e136dbc0b",
       "dermatology": "c8852ba546e92249debecd9a897cda3f94fd10872fa13c62eb14ab40e279166e",
       "spectf": "664081130aa48fc189b2e1d3c44332f0b3eb1ee6d31b79af1643c137697677e4",
       "backache": "21d506c397dfeb3edbbbc253b923f59be6edb516689677ebef535296c6c62242"}
for short in TABS:
    name = TAB[short]; path = f"{DATA_DIR}/{name}.tsv.gz"
    if not os.path.exists(path): urllib.request.urlretrieve(PMLB.format(name), path)
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()
    if sha != SHA[name]: print(f"WARNING: {name} checksum differs from the file used to write this notebook ({sha[:12]}...)")
    df = pd.read_csv(path, sep="\t")
    DATA[short] = (df.drop(columns="target").values.astype(float), pd.factorize(df["target"], sort=True)[0])

pd.DataFrame([{"dataset": n, "samples": X.shape[0], "features": X.shape[1], "classes": len(np.unique(y)),
               "smallest class": int(np.unique(y, return_counts=True)[1].min())} for n, (X, y) in DATA.items()])

,dataset,samples,features,classes,smallest class
0,colon,62,2000,2,22
1,ALLAML,72,7129,2,25
2,lung,203,3312,5,6
3,GLIOMA,50,4434,4,7
4,GSE93272,101,54613,2,35
5,WDBC,569,30,2,212
6,Hepatitis,155,19,2,32
7,Dermatology,366,34,6,20
8,SPECTF,349,44,2,95
9,Backache,180,32,2,25


## 4. Fitness and test evaluation (identical to Notebook 7; earlier gene caches are reused because the fitness is deterministic)

In [4]:
def mask_str(m): return np.packbits(np.asarray(m, dtype=np.uint8)).tobytes().hex()
NEW_EVAL = {}                                   # dataset -> [number of new evaluations, seconds spent on them]

def use_dataset(name, seed):
    global X_tr, X_te, y_tr, y_te, N_FEATURES, _cache, CV, CUR, SCORE
    X, y = DATA[name]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    if name == "GSE93272":
        col_mean = np.nanmean(X_tr, axis=0)
        X_tr = np.where(np.isnan(X_tr), col_mean, X_tr); X_te = np.where(np.isnan(X_te), col_mean, X_te)
        top = np.argsort(X_tr.var(axis=0))[-N_TOP_RA:]; X_tr, X_te = X_tr[:, top], X_te[:, top]
    sc = MinMaxScaler().fit(X_tr); X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
    N_FEATURES = X_tr.shape[1]
    CV = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=seed); SCORE = SCORING[name]; CUR = (name, seed)
    _cache = {}
    for d in PREV_DIRS + [OUT_DIR]:
        cf = f"{d}/cache_{name}_s{seed}.pkl"
        if os.path.exists(cf): _cache.update(pickle.load(open(cf, "rb")))

def save_cache_now(): pickle.dump(_cache, open(f"{OUT_DIR}/cache_{CUR[0]}_s{CUR[1]}.pkl", "wb"))

def fitness_function(mask):
    key = mask_str(mask)
    if key in _cache: return _cache[key]["fit"]
    sel = np.flatnonzero(mask)
    if len(sel) == 0:
        _cache[key] = {"fit": 1.0, "acc": 0.0}; return 1.0
    t0 = time.perf_counter()
    acc = cross_val_score(KNeighborsClassifier(KNN_K), X_tr[:, sel], y_tr, cv=CV, scoring=SCORE).mean()
    f = ALPHA * (1 - acc) + BETA * len(sel) / len(mask)
    _cache[key] = {"fit": f, "acc": acc}
    c = NEW_EVAL.setdefault(CUR[0], [0, 0.0]); c[0] += 1; c[1] += time.perf_counter() - t0
    return f

def test_metrics(mask):
    sel = np.flatnonzero(mask)
    p = KNeighborsClassifier(KNN_K).fit(X_tr[:, sel], y_tr).predict(X_te[:, sel])
    acc = balanced_accuracy_score(y_te, p) if SCORE == "balanced_accuracy" else accuracy_score(y_te, p)
    return {"test_acc": float(acc), "test_prec": float(precision_score(y_te, p, average="macro", zero_division=0)),
            "test_rec": float(recall_score(y_te, p, average="macro", zero_division=0)), "test_f1": float(f1_score(y_te, p, average="macro"))}
print("Fitness ready.")

Fitness ready.


## 5. Algorithms: the Notebook 7 code plus the Q-learning controller, the GA+SI / HO+SI add-on, fitness timing and the filters

In [5]:
# ===================== RL-DOHO v2: shared algorithm code =====================
# Every method gets the same number of FITNESS EVALUATIONS (not iterations), because the
# published HO evaluates about 3N candidates per iteration while DO evaluates N.
# Encoding: continuous position x in [LB, UB]^D, feature j selected if sigmoid(x_j) > 0.5 (i.e. x_j > 0).
import math, time
import numpy as np, pandas as pd

LB, UB = -3.0, 3.0

class BudgetExhausted(Exception):
    pass

def binarize(x):
    return (np.asarray(x) > 0).astype(np.int8)          # identical to sigmoid(x) > 0.5

def mkey(mask):
    return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()

class Problem:
    """Wraps a (cached, deterministic) fitness function and enforces the evaluation budget.
    Also records the time spent inside the fitness function (fsec) and the number of distinct
    subsets evaluated (len(seen)), so the algorithm's own overhead can be separated from fitness cost."""
    def __init__(self, fitness_fn, D, budget):
        self.fitness_fn, self.D, self.budget, self.calls = fitness_fn, D, budget, 0
        self.fsec, self.seen = 0.0, set()
    def left(self):
        return self.budget - self.calls
    def f(self, mask):
        if self.calls >= self.budget:
            raise BudgetExhausted
        self.calls += 1
        m = np.asarray(mask, dtype=np.int8); self.seen.add(mkey(m))
        t0 = time.perf_counter(); v = float(self.fitness_fn(m)); self.fsec += time.perf_counter() - t0
        return v
    def fx(self, x):
        return self.f(binarize(x))

# ------------------------------------------------------------------ initialisation
def init_population(rng, N, D, mode="dense"):
    """dense : x ~ U(-1, 1)  -> each feature selected with probability 0.5 (as in Notebooks 1-5).
       sparse: agent i keeps a fraction p_i ~ log-uniform[min(10/D, 0.1), 0.1] of the features
               (used when D > 100: ten features up to 10% of them)."""
    if mode == "dense":
        return rng.uniform(-1, 1, (N, D))
    lo, hi = math.log(min(10.0 / D, 0.1)), math.log(0.1)
    X = np.empty((N, D))
    for i in range(N):
        p = math.exp(rng.uniform(lo, hi))
        sel = rng.random(D) < p
        if not sel.any():
            sel[rng.integers(D)] = True
        X[i] = np.where(sel, rng.uniform(0.1, 1.0, D), -rng.uniform(0.1, 1.0, D))
    return X

def random_position(rng, D, mode):
    return init_population(rng, 1, D, mode)[0]

# ------------------------------------------------------------------ Levy flight (Mantegna)
def levy(rng, shape, beta=1.5):
    num = math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
    den = math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2)
    sigma = (num / den) ** (1 / beta)
    u = rng.normal(0, sigma, shape); v = rng.normal(0, 1, shape)
    return u / np.abs(v) ** (1 / beta)

def lognpdf(x):                                          # lognormal pdf, mu = 0, sigma = 1
    x = np.maximum(x, 1e-300)
    return np.exp(-np.log(x) ** 2 / 2) / (x * math.sqrt(2 * math.pi))

# ------------------------------------------------------------------ Dandelion Optimizer (Zhao et al., 2022)
def do_move(rng, X, elite, t, T):
    """One full DO iteration: rising (Eqs. 2-8), descending (Eqs. 10-11), landing (Eqs. 12-14).
    Returns the new population; the caller evaluates it (N evaluations)."""
    N, D = X.shape
    T = max(T, 2); t = min(max(t, 1), T)
    alpha = rng.random() * ((t / T) ** 2 - 2 * t / T + 1)                     # Eq. 4: adaptive step
    a = 1.0 / (T ** 2 - 2 * T + 1); b = -2 * a; c = 1 - a - b
    k = 1 - rng.random() * (c + a * t ** 2 + b * t)                          # Eq. 8 (k with q)
    # --- rising stage
    if rng.standard_normal() < 1.5:                                          # clear weather
        lamb = np.abs(rng.standard_normal((N, D)))
        theta = (2 * rng.random(N) - 1) * math.pi
        r = 1 / np.exp(theta); vx = r * np.cos(theta); vy = r * np.sin(theta)
        Xs = rng.uniform(LB, UB, (N, D))                                     # Eq. 3: random position
        X1 = X + alpha * (vx * vy)[:, None] * lognpdf(lamb) * (Xs - X)       # Eq. 2
    else:                                                                    # rainy weather
        X1 = X * k                                                           # Eq. 7
    X1 = np.clip(X1, LB, UB)
    # --- descending stage (Brownian motion around the population mean)
    beta = rng.standard_normal((N, D))
    Xm = X1.mean(axis=0)                                                     # Eq. 11
    X2 = np.clip(X1 - beta * alpha * (Xm - beta * alpha * X1), LB, UB)      # Eq. 10
    # --- landing stage (Levy flight around the elite)
    step = 0.01 * levy(rng, (N, D))                                          # Eq. 13, s = 0.01, beta = 1.5
    delta = 2 * t / T                                                        # Eq. 14
    X3 = elite + step * alpha * (elite - X2 * delta)                         # Eq. 12
    return np.clip(X3, LB, UB)

# ------------------------------------------------------------------ Hippopotamus Optimization (Amiri et al., 2024)
def ho_iteration(rng, X, F, dominant, t, T, prob, on_eval):
    """One full HO iteration with greedy selection, updating X, F in place.
    Phase 1 (first half): male + female/immature candidates (Eqs. 3-9)      -> 2 evaluations per agent
    Phase 2 (second half): predator + defence candidate (Eqs. 10-16)        -> 2 evaluations per agent
    Phase 3 (all agents): escape within shrinking local bounds (Eqs. 17-19) -> 1 evaluation per agent
    on_eval(x, f) is called for every evaluated candidate that is a hippopotamus position."""
    N, D = X.shape
    half = N // 2
    T = max(T, 1); t = min(max(t, 1), T)
    for i in range(half):
        I1, I2 = rng.integers(1, 3, 2); rho = rng.integers(0, 2, 2)
        g = int(rng.integers(1, N + 1)); grp = rng.choice(N, g, replace=False)
        MG = X[grp].mean(axis=0)
        h = [I2 * rng.random(D) + (1 - rho[0]), 2 * rng.random(D) - 1, rng.random(D),
             I1 * rng.random(D) + (1 - rho[1]), rng.random()]                  # Eq. 4
        h1, h2 = h[rng.integers(5)], h[rng.integers(5)]
        x_m = np.clip(X[i] + rng.random() * (dominant - I1 * X[i]), LB, UB)     # Eq. 3 (male)
        if math.exp(-t / T) > 0.6:                                            # Eq. 5
            x_f = X[i] + h1 * (dominant - I2 * MG)                            # Eq. 6
        elif rng.random() > 0.5:
            x_f = X[i] + h2 * (MG - dominant)                                 # Eq. 7
        else:
            x_f = rng.uniform(LB, UB, D)
        x_f = np.clip(x_f, LB, UB)
        for x in (x_m, x_f):                                                  # Eqs. 8-9: greedy
            f = prob.fx(x); on_eval(x, f)
            if f < F[i]: X[i], F[i] = x, f
    for i in range(half, N):
        pred = rng.uniform(LB, UB, D)                                         # Eq. 10
        f_pred = prob.fx(pred)
        dist = np.abs(pred - X[i]) + 1e-12                                    # Eq. 11
        b_, c_, d_ = rng.uniform(2, 4), rng.uniform(1, 1.5), rng.uniform(2, 3)
        l_ = rng.uniform(-2 * math.pi, 2 * math.pi)                           # 2*pi*g with g in [-1, 1]
        RL = 0.05 * levy(rng, D)                                              # Eq. 13
        if F[i] > f_pred:
            x = RL * pred + (b_ / (c_ - d_ * math.cos(l_))) * (1 / dist)      # Eq. 12, case 1
        else:
            x = RL * pred + (b_ / (c_ - d_ * math.cos(l_))) * (1 / (2 * dist + rng.random(D)))
        x = np.clip(x, LB, UB)
        f = prob.fx(x); on_eval(x, f)
        if f < F[i]: X[i], F[i] = x, f                                        # Eqs. 15-16
    lo, hi = LB / t, UB / t                                                   # Eq. 17: local bounds
    for i in range(N):
        s = [2 * rng.random(D) - 1, rng.random(), rng.standard_normal()][rng.integers(3)]   # Eq. 19
        x = np.clip(X[i] + rng.random() * (lo + s * (hi - lo)), LB, UB)      # Eq. 18
        f = prob.fx(x); on_eval(x, f)
        if f < F[i]: X[i], F[i] = x, f                                        # Eq. 20

# ------------------------------------------------------------------ run bookkeeping
class Tracker:
    """Tracks the best subset found and logs one row per iteration."""
    def __init__(self, name, prob, verbose=False, tie_eps=0.0):
        self.name, self.prob, self.verbose, self.tie_eps = name, prob, verbose, tie_eps
        self.best_f, self.best_x, self.best_m = np.inf, None, None
        self.rows, self.t0 = [], time.time()
    def better(self, f, nf):
        if self.best_m is None: return True
        bf, bn = self.best_f, int(self.best_m.sum())
        e = self.tie_eps
        if e <= 0: return f < bf
        return f < bf - e or (abs(f - bf) <= e and (nf < bn or (nf == bn and f < bf)))
    def offer(self, x, f, mask=None):
        m = binarize(x) if mask is None else mask
        if self.better(f, int(m.sum())):
            self.best_f, self.best_x, self.best_m = f, (None if x is None else np.array(x, float)), m.copy()
            return True
        return False
    def log(self, it, action, F=None, reward=None):
        row = dict(iter=it, evals=self.prob.calls, action=action, best=self.best_f,
                   n_feats=int(self.best_m.sum()), pop_mean=(float(np.mean(F)) if F is not None else np.nan),
                   reward=reward, sec=time.time() - self.t0)
        self.rows.append(row)
        if self.verbose:
            rw = "" if reward is None else f"  r={reward:.3f}"
            print(f"  {self.name:<16} it {it:>3}  evals {self.prob.calls:>5}/{self.prob.budget}  {action:<10}"
                  f" best {self.best_f:.5f}  feats {row['n_feats']:>5}{rw}")
    def result(self):
        return {"mask": self.best_m, "fit": self.best_f, "hist": pd.DataFrame(self.rows)}

def _start(prob, rng, N, init, tr):
    X = init_population(rng, N, prob.D, init)
    F = np.array([prob.fx(x) for x in X])
    for x, f in zip(X, F): tr.offer(x, f)
    tr.log(0, "init", F)
    return X, F

# ------------------------------------------------------------------ DO, HO, DOHO (published update rules)
def run_do(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("DO", prob, verbose)
    T = max(2, (prob.budget - N) // N)
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            X = do_move(rng, X, tr.best_x, t, T)
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            tr.log(t, "DO", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_ho(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("HO", prob, verbose)
    T = max(1, math.ceil((prob.budget - N) / (3 * N)))
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            dom = X[np.argmin(F)].copy()
            ho_iteration(rng, X, F, dom, t, T, prob, tr.offer)
            tr.log(t, "HO", F)
    except BudgetExhausted:
        if tr.rows and tr.rows[-1]["evals"] != prob.calls: tr.log(len(tr.rows), "HO (partial)", F)
    return tr.result()

def run_doho(prob, N, seed, init="dense", verbose=False):
    """Fixed switch: published DO for the first half of the budget, then published HO."""
    rng = np.random.default_rng(seed); tr = Tracker("DOHO", prob, verbose)
    half_budget = prob.budget // 2
    T1 = max(2, (half_budget - N) // N); T2 = max(1, math.ceil((prob.budget - half_budget) / (3 * N)))
    it = 0
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T1 + 1):
            if prob.calls + N > half_budget: break
            X = do_move(rng, X, tr.best_x, t, T1)
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            it += 1; tr.log(it, "DO", F)
        for t in range(1, T2 + 1):
            ho_iteration(rng, X, F, X[np.argmin(F)].copy(), t, T2, prob, tr.offer)
            it += 1; tr.log(it, "HO", F)
    except BudgetExhausted:
        pass
    return tr.result()

# ------------------------------------------------------------------ controllers
class UCB1:
    def __init__(self, n_arms, seed, c=1.0):
        self.rng = np.random.default_rng(seed + 7919); self.c = c
        self.n = np.zeros(n_arms); self.v = np.zeros(n_arms)
    def choose(self, state=None):
        untried = np.flatnonzero(self.n == 0)
        if len(untried): return int(self.rng.choice(untried)), "i"
        ucb = self.v + self.c * np.sqrt(2 * np.log(self.n.sum()) / self.n)
        return int(self.rng.choice(np.flatnonzero(np.isclose(ucb, ucb.max())))), "u"
    def update(self, a, r):
        self.n[a] += 1; self.v[a] += (r - self.v[a]) / self.n[a]
    def finish(self):
        pass

class RandomArm(UCB1):
    def choose(self, state=None): return int(self.rng.integers(len(self.n))), "r"

class FixedArm(UCB1):
    def __init__(self, n_arms, seed, arm=0):
        super().__init__(n_arms, seed); self.arm = arm
    def choose(self, state=None): return self.arm, "f"

N_STATES = 6
def controller_state(calls, budget, last_gain):
    """State for the Q-learning controller: budget phase (first, second, last third) x whether the
    previous intervention improved the best subset (0/1) -> 6 states."""
    phase = min(2, int(3 * calls / budget))
    return 2 * phase + (1 if last_gain > 0 else 0)

class QLearn(UCB1):
    """Tabular one-step Q-learning over the arms. Epsilon-greedy (eps = 0.2), learning rate 0.3,
    discount 0.5, Q initialised at 0. The next state is the state at the next decision; the last
    decision of a run is updated without bootstrapping. Settings fixed in advance, not tuned."""
    def __init__(self, n_arms, seed, n_states=N_STATES, lr=0.3, gamma=0.5, eps=0.2):
        super().__init__(n_arms, seed)
        self.Q = np.zeros((n_states, n_arms)); self.lr, self.gamma, self.eps = lr, gamma, eps
        self.pending, self.s = None, 0
    def choose(self, state=0):
        if self.pending is not None:
            s, a, r = self.pending
            self.Q[s, a] += self.lr * (r + self.gamma * self.Q[state].max() - self.Q[s, a]); self.pending = None
        self.s = state
        if self.rng.random() < self.eps:
            return int(self.rng.integers(self.Q.shape[1])), "e"
        q = self.Q[state]
        return int(self.rng.choice(np.flatnonzero(np.isclose(q, q.max())))), "q"
    def update(self, a, r):
        self.n[a] += 1; self.pending = (self.s, a, r)
    def finish(self):
        if self.pending is not None:
            s, a, r = self.pending; self.Q[s, a] += self.lr * (r - self.Q[s, a]); self.pending = None

CONTROLLERS = {"ucb": UCB1, "random": RandomArm, "qlearn": QLearn}

ARMS = ("PERTURB", "RESTART", "DO", "HO")

# ------------------------------------------------------------------ RL-DOHO v2
def run_rldoho(prob, N, seed, init="dense", verbose=False, controller="ucb", reward="gain",
               kappa=2, tie_eps=1e-3, k_max=3, tabu_tries=20, intervene=True, name="RL-DOHO"):
    """DO (published) while the best improves; after `kappa` iterations without an accepted improvement,
    a controller picks one arm: PERTURB / RESTART (worse half, tabu) or one published DO / HO iteration.
    reward="gain": 0.8 * min(1, g / g_max) + 0.2 * share, where g = fitness gain of the best per evaluation
                   spent (parsimony-only moves give g = 0) and g_max = largest g seen so far in the run.
    reward="old" : 1 if the best changed, otherwise 0.2 * share (Notebooks 1-5)."""
    rng = np.random.default_rng(seed); D = prob.D
    tr = Tracker(name, prob, verbose, tie_eps=tie_eps)
    ctrl = CONTROLLERS.get(controller)
    ctrl = ctrl(len(ARMS), seed) if ctrl else FixedArm(len(ARMS), seed, arm=ARMS.index(controller))
    T_do = max(2, (prob.budget - N) // N); T_ho = max(1, math.ceil((prob.budget - N) / (3 * N)))
    def sched(T):                     # DO / HO schedules follow the share of the budget already spent
        return min(T, max(1, math.ceil(T * prob.calls / prob.budget)))
    tabu = set(); g_max = 0.0; stag = 0; it = 0; last_gain = 0.0
    arm_log = []
    def offer(x, f):
        tabu.add(mkey(binarize(x)))
        return tr.offer(x, f)
    def tabu_draw(make):
        for _ in range(tabu_tries):
            x = np.clip(make(), LB, UB)
            if mkey(binarize(x)) not in tabu: break
        return x
    try:
        X = init_population(rng, N, D, init)
        F = np.array([prob.fx(x) for x in X])
        for x, f in zip(X, F): offer(x, f)
        tr.log(0, "init", F)
        while prob.left() > 0:
            it += 1
            prev_best, prev_feats, prev_med, calls0 = tr.best_f, int(tr.best_m.sum()), float(np.median(F)), prob.calls
            changed_before = (tr.best_f, int(tr.best_m.sum()))
            if not intervene or stag < kappa:
                arm, how = None, None
                X = do_move(rng, X, tr.best_x, sched(T_do), T_do)
                F = np.array([prob.fx(x) for x in X])
                for x, f in zip(X, F): offer(x, f)
                moved = np.arange(N)
            else:
                arm, how = ctrl.choose(controller_state(prob.calls, prob.budget, last_gain)); a = ARMS[arm]
                if a == "DO":
                    X = do_move(rng, X, tr.best_x, sched(T_do), T_do)
                    F = np.array([prob.fx(x) for x in X])
                    for x, f in zip(X, F): offer(x, f)
                    moved = np.arange(N)
                elif a == "HO":
                    F_before = F.copy()
                    ho_iteration(rng, X, F, X[np.argmin(F)].copy(), sched(T_ho), T_ho, prob, offer)
                    moved = np.flatnonzero(F != F_before)
                else:
                    worst = np.argsort(F)[N // 2:]
                    for i in worst:
                        if a == "PERTURB":
                            def make():
                                x = tr.best_x.copy(); k = int(rng.integers(1, k_max + 1))
                                j = rng.choice(D, k, replace=False)
                                x[j] = -np.sign(x[j] + 1e-12) * rng.uniform(0.5, 2.0, k)
                                return x
                        else:
                            make = lambda: random_position(rng, D, init)
                        X[i] = tabu_draw(make)
                        F[i] = prob.fx(X[i]); offer(X[i], F[i])
                    moved = worst
            cost = max(1, prob.calls - calls0)
            changed = (tr.best_f, int(tr.best_m.sum())) != changed_before
            stag = 0 if changed else stag + 1
            g = max(0.0, prev_best - tr.best_f) / cost
            g_max = max(g_max, g)
            if arm is None:
                tr.log(it, "explore", F)
            else:
                share = float(np.mean(F[moved] < prev_med)) if len(moved) else 0.0
                if reward == "gain":
                    r = 0.8 * (min(1.0, g / g_max) if g_max > 0 else 0.0) + 0.2 * share
                else:
                    r = 1.0 if changed else 0.2 * share
                ctrl.update(arm, r); last_gain = prev_best - tr.best_f
                arm_log.append((it, ARMS[arm], how, r, prev_best - tr.best_f, cost))
                tr.log(it, f"{ARMS[arm]}({how})", F, reward=r)
    except BudgetExhausted:
        pass
    ctrl.finish()
    res = tr.result()
    res["arms"] = pd.DataFrame(arm_log, columns=["iter", "arm", "how", "reward", "gain", "cost"])
    return res

# ------------------------------------------------------------------ GA, BPSO, BGWO (same budget and initial population)
def run_ga(prob, N, seed, init="dense", verbose=False, pc=0.9, tour=3):
    rng = np.random.default_rng(seed); tr = Tracker("GA", prob, verbose); D = prob.D; pm = 1.0 / D
    try:
        X, F = _start(prob, rng, N, init, tr)
        M = binarize(X); it = 0
        while True:
            def pick():
                idx = rng.choice(N, tour, replace=False); return M[idx[np.argmin(F[idx])]]
            new = [tr.best_m.copy()]
            while len(new) < N:
                p1, p2 = pick(), pick()
                if rng.random() < pc:
                    cm = rng.random(D) < 0.5; kids = [np.where(cm, p1, p2), np.where(cm, p2, p1)]
                else:
                    kids = [p1.copy(), p2.copy()]
                for c in kids:
                    c = np.where(rng.random(D) < pm, 1 - c, c).astype(np.int8)
                    if len(new) < N: new.append(c)
            M = np.array(new, dtype=np.int8)
            F = np.empty(N)
            for i in range(N):
                F[i] = prob.f(M[i]); tr.offer(None, F[i], mask=M[i])
            it += 1; tr.log(it, "gen", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_bpso(prob, N, seed, init="dense", verbose=False, c1=2.0, c2=2.0, vmax=6.0):
    rng = np.random.default_rng(seed); tr = Tracker("BPSO", prob, verbose); D = prob.D
    T = max(2, (prob.budget - N) // N)
    try:
        X0, F = _start(prob, rng, N, init, tr)
        X = binarize(X0); V = rng.uniform(-1, 1, (N, D))
        if init == "sparse":                      # start velocities consistent with the sparse masks
            V = np.where(X == 1, rng.uniform(0, 1, (N, D)), rng.uniform(-6, -1, (N, D)))
        pb, pbf = X.copy(), F.copy()
        for t in range(1, T + 1):
            w = 0.9 - 0.5 * t / T
            r1, r2 = rng.random((N, D)), rng.random((N, D))
            V = np.clip(w * V + c1 * r1 * (pb - X) + c2 * r2 * (tr.best_m - X), -vmax, vmax)
            X = (rng.random((N, D)) < 1 / (1 + np.exp(-V))).astype(np.int8)
            F = np.empty(N)
            for i in range(N):
                F[i] = prob.f(X[i]); tr.offer(None, F[i], mask=X[i])
            imp = F < pbf; pb[imp] = X[imp]; pbf[imp] = F[imp]
            tr.log(t, "move", F)
    except BudgetExhausted:
        pass
    return tr.result()

def run_bgwo(prob, N, seed, init="dense", verbose=False):
    rng = np.random.default_rng(seed); tr = Tracker("BGWO", prob, verbose); D = prob.D
    T = max(2, (prob.budget - N) // N)
    try:
        X, F = _start(prob, rng, N, init, tr)
        for t in range(1, T + 1):
            a = 2 - 2 * t / T
            leaders = X[np.argsort(F)[:3]].copy()
            new = np.empty_like(X)
            for i in range(N):
                xs = [Lp - (2 * a * rng.random(D) - a) * np.abs(2 * rng.random(D) * Lp - X[i]) for Lp in leaders]
                new[i] = np.clip(np.mean(xs, axis=0), LB, UB)
            X = new
            F = np.array([prob.fx(x) for x in X])
            for x, f in zip(X, F): tr.offer(x, f)
            tr.log(t, "move", F)
    except BudgetExhausted:
        pass
    return tr.result()


# ------------------------------------------------------------------ stagnation-triggered add-on for any base optimiser
def ga_generation(rng, X, F, best_m, prob, offer, pc=0.9, tour=3):
    """One GA generation (tournament 3, uniform crossover, bit-flip 1/D, elitism of 1) on the masks of X.
    Children are stored as positions +-1 so that the shared PERTURB / RESTART operators can act on them."""
    N, D = X.shape; M = binarize(X); pm = 1.0 / D
    def pick():
        idx = rng.choice(N, tour, replace=False); return M[idx[np.argmin(F[idx])]]
    new = [best_m.copy()]
    while len(new) < N:
        p1, p2 = pick(), pick()
        if rng.random() < pc:
            cm = rng.random(D) < 0.5; kids = [np.where(cm, p1, p2), np.where(cm, p2, p1)]
        else:
            kids = [p1.copy(), p2.copy()]
        for c in kids:
            c = np.where(rng.random(D) < pm, 1 - c, c).astype(np.int8)
            if len(new) < N: new.append(c)
    X = np.where(np.array(new) == 1, 1.0, -1.0)
    F = np.empty(N)
    for i in range(N):
        F[i] = prob.fx(X[i]); offer(X[i], F[i])
    return X, F

def run_si(prob, N, seed, base="GA", init="dense", verbose=False, controller="ucb", reward="gain",
           kappa=2, tie_eps=1e-3, k_max=3, tabu_tries=20, name=None):
    """The RL-DOHO intervention layer around another optimiser ("GA" or "HO"): the base optimiser runs while
    the best subset improves; after `kappa` stalled iterations the controller picks PERTURB, RESTART or one
    base iteration, with the same gain reward, tabu memory and tie-break as RL-DOHO. Used to test whether the
    interventions help other optimisers too."""
    rng = np.random.default_rng(seed); D = prob.D; arms = ("PERTURB", "RESTART", base)
    tr = Tracker(name or f"{base}+SI", prob, verbose, tie_eps=tie_eps)
    ctrl = CONTROLLERS[controller](len(arms), seed)
    T_ho = max(1, math.ceil((prob.budget - N) / (3 * N)))
    sched = lambda T: min(T, max(1, math.ceil(T * prob.calls / prob.budget)))
    tabu = set(); g_max = 0.0; stag = 0; it = 0; last_gain = 0.0; arm_log = []
    def offer(x, f):
        tabu.add(mkey(binarize(x))); return tr.offer(x, f)
    def base_step(X, F):
        if base == "GA":
            X, F = ga_generation(rng, X, F, tr.best_m, prob, offer); return X, F, np.arange(N)
        F0 = F.copy(); ho_iteration(rng, X, F, X[np.argmin(F)].copy(), sched(T_ho), T_ho, prob, offer)
        return X, F, np.flatnonzero(F != F0)
    try:
        X = init_population(rng, N, D, init)
        F = np.array([prob.fx(x) for x in X])
        for x, f in zip(X, F): offer(x, f)
        tr.log(0, "init", F)
        while prob.left() > 0:
            it += 1
            prev_best, prev_med, calls0 = tr.best_f, float(np.median(F)), prob.calls
            before = (tr.best_f, int(tr.best_m.sum()))
            if stag < kappa:
                arm = None; X, F, moved = base_step(X, F)
            else:
                arm, how = ctrl.choose(controller_state(prob.calls, prob.budget, last_gain)); a = arms[arm]
                if a == base:
                    X, F, moved = base_step(X, F)
                else:
                    worst = np.argsort(F)[N // 2:]
                    for i in worst:
                        if a == "PERTURB":
                            def make():
                                x = tr.best_x.copy(); k = int(rng.integers(1, k_max + 1))
                                j = rng.choice(D, k, replace=False)
                                x[j] = -np.sign(x[j] + 1e-12) * rng.uniform(0.5, 2.0, k)
                                return x
                        else:
                            make = lambda: random_position(rng, D, init)
                        for _ in range(tabu_tries):
                            x = np.clip(make(), LB, UB)
                            if mkey(binarize(x)) not in tabu: break
                        X[i] = x; F[i] = prob.fx(x); offer(x, F[i])
                    moved = worst
            cost = max(1, prob.calls - calls0)
            changed = (tr.best_f, int(tr.best_m.sum())) != before
            stag = 0 if changed else stag + 1
            g = max(0.0, prev_best - tr.best_f) / cost; g_max = max(g_max, g)
            if arm is None:
                tr.log(it, "explore", F)
            else:
                share = float(np.mean(F[moved] < prev_med)) if len(moved) else 0.0
                r = (0.8 * (min(1.0, g / g_max) if g_max > 0 else 0.0) + 0.2 * share) if reward == "gain" \
                    else (1.0 if changed else 0.2 * share)
                ctrl.update(arm, r); last_gain = prev_best - tr.best_f
                arm_log.append((it, arms[arm], how, r, prev_best - tr.best_f, cost))
                tr.log(it, f"{arms[arm]}({how})", F, reward=r)
    except BudgetExhausted:
        pass
    ctrl.finish()
    res = tr.result(); res["arms"] = pd.DataFrame(arm_log, columns=["iter", "arm", "how", "reward", "gain", "cost"])
    return res

def run_ga_si(prob, N, seed, init="dense", verbose=False, **kw): return run_si(prob, N, seed, "GA", init, verbose, **kw)
def run_ho_si(prob, N, seed, init="dense", verbose=False, **kw): return run_si(prob, N, seed, "HO", init, verbose, **kw)

# ------------------------------------------------------------------ filter / embedded baselines (as in Notebooks 4 and 5)
# Each ranks features on the training data only; k is then chosen by the same fitness (CV on training data).
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif

def rank_lasso(X, y, seed, k_max):
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-12)
    m = LogisticRegression(penalty="l1", solver="saga", C=0.5, max_iter=3000, random_state=seed).fit(Xs, y)
    return np.argsort(-np.abs(m.coef_).sum(0), kind="stable")

def rank_mrmr(X, y, seed, k_max):
    rel = mutual_info_classif(X, y, random_state=seed)
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-12); n = len(Xs)
    sel = [int(np.argmax(rel))]; red = np.zeros(X.shape[1])
    for _ in range(min(k_max, X.shape[1], 2000) - 1):
        red += np.abs(Xs.T @ Xs[:, sel[-1]]) / n
        score = rel - red / len(sel); score[sel] = -np.inf
        sel.append(int(np.argmax(score)))
    rest = [j for j in np.argsort(-rel) if j not in set(sel)]
    return np.array(sel + rest)

def rank_relieff(X, y, seed, k_max, m=500, k=10):
    rng = np.random.default_rng(seed)
    Xn = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-12)
    classes, counts = np.unique(y, return_counts=True); prior = dict(zip(classes, counts / len(y)))
    idx = rng.choice(len(Xn), min(m, len(Xn)), replace=False); W = np.zeros(Xn.shape[1])
    for i in idx:
        d = np.abs(Xn - Xn[i]).sum(1); d[i] = np.inf
        for c in classes:
            cand = np.flatnonzero(y == c); cand = cand[cand != i]
            if len(cand) == 0: continue
            nn = cand[np.argsort(d[cand])[:k]]
            diff = np.abs(Xn[nn] - Xn[i]).mean(0)
            if c == y[i]: W -= diff / len(idx)
            else: W += prior[c] / (1 - prior[y[i]]) * diff / len(idx)
    return np.argsort(-W, kind="stable")

RANKERS = {"LASSO": rank_lasso, "mRMR": rank_mrmr, "ReliefF": rank_relieff}

def k_grid(D):
    if D <= 40: return list(range(1, D + 1))
    return sorted({k for k in [5, 10, 20, 50, 100, 200, 500, 1000, 2000, D] if k <= D})

def run_filter(name, seed, X_rank, y_rank, fitness_fn, D):
    """Returns (mask, fitness, chosen k). 'All features' keeps everything."""
    if name == "All features":
        m = np.ones(D, np.int8); return m, fitness_fn(m), D
    ks = k_grid(D); order = RANKERS[name](np.asarray(X_rank, float), np.asarray(y_rank), seed, max(ks))
    best = None
    for k in ks:
        m = np.zeros(D, np.int8); m[order[:k]] = 1
        f = fitness_fn(m)
        if best is None or f < best[1] - 1e-12: best = (m, f, k)
    return best

RUNNERS = {"DO": run_do, "HO": run_ho, "DOHO": run_doho, "RL-DOHO": run_rldoho,
           "GA": run_ga, "BPSO": run_bpso, "BGWO": run_bgwo, "GA+SI": run_ga_si, "HO+SI": run_ho_si}

print("Algorithms ready:", list(RUNNERS), "| controllers:", list(CONTROLLERS))

Algorithms ready: ['DO', 'HO', 'DOHO', 'RL-DOHO', 'GA', 'BPSO', 'BGWO', 'GA+SI', 'HO+SI'] | controllers: ['ucb', 'random', 'qlearn']


In [6]:
# ================= Statistics helpers =================
from scipy.stats import wilcoxon, friedmanchisquare, binomtest
TIE = 1e-12

def holm(ps):
    ps = np.asarray(ps, float); order = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for i, j in enumerate(order):
        run = max(run, (m - i) * ps[j]); adj[j] = min(1.0, run)
    return adj

def wtl(a, b, higher_better):
    d = np.asarray(a) - np.asarray(b)
    w = int(np.sum(d > TIE) if higher_better else np.sum(d < -TIE)); t = int(np.sum(np.abs(d) <= TIE))
    return w, t, len(d) - w - t

def per_dataset_tests(R, ref, methods, metric, higher_better):
    """Friedman over `methods` (seeds as blocks) and Holm-corrected Wilcoxon of `ref` vs each other method."""
    rows = []
    for ds in R.dataset.unique():
        P = R[R.dataset == ds].pivot_table(index="seed", columns="method", values=metric)[methods].dropna()
        try: fp = friedmanchisquare(*[P[c] for c in methods]).pvalue
        except ValueError: fp = np.nan
        tmp = []
        for c in methods:
            if c == ref: continue
            w, t, l = wtl(P[ref], P[c], higher_better)
            try: p = wilcoxon(P[ref], P[c], zero_method="zsplit").pvalue
            except ValueError: p = 1.0
            tmp.append(dict(dataset=ds, metric=metric, vs=c, W=w, T=t, L=l, mean_diff=float((P[ref] - P[c]).mean()), p=p, friedman_p=fp))
        for r_, h in zip(tmp, holm([x["p"] for x in tmp])): r_["holm_p"] = h; rows.append(r_)
    return pd.DataFrame(rows)

def pooled_sign(R, ref, others, metric, higher_better):
    rows = []
    A = R[R.method == ref].set_index(["dataset", "seed"])[metric]
    for c in others:
        B = R[R.method == c].set_index(["dataset", "seed"])[metric]
        a, b = A.align(B, join="inner")
        w, t, l = wtl(a, b, higher_better)
        p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
        rows.append(dict(metric=metric, vs=c, W=w, T=t, L=l, n=len(a), sign_p=p))
    out = pd.DataFrame(rows); out["holm_p"] = holm(out.sign_p.values) if len(out) else []
    return out

def mean_ranks(R, methods, metric, higher_better):
    X = R[R.method.isin(methods)].copy()
    X["r"] = X.groupby(["dataset", "seed"])[metric].rank(ascending=not higher_better)
    return X.pivot_table(index="method", columns="dataset", values="r").reindex(methods)

def summary_table(R, methods):
    g = R[R.method.isin(methods)].groupby(["dataset", "method"])
    S = g.agg(fit=("fit", "mean"), fit_sd=("fit", "std"), cv=("cv", "mean"), test_acc=("test_acc", "mean"),
              acc_sd=("test_acc", "std"), test_f1=("test_f1", "mean"), n_feats=("n_feats", "mean")).reset_index()
    return S
print("Statistics helpers ready.")

def ablation_report(A, main, variants, label):
    """A: DataFrame dataset, variant, seed, fit, ... ; compares `main` with every other variant."""
    A = A.copy(); A["r"] = A.groupby(["dataset", "seed"]).fit.rank()
    print(f"\n=== {label}: mean per-run rank on fitness (1 = best) ===")
    print(A.pivot_table(index="variant", columns="dataset", values="r").reindex(variants).round(2).assign(
        ALL=A.groupby("variant").r.mean().reindex(variants).round(2)).to_string())
    print(f"\nMean fitness / test accuracy / features:")
    print(A.groupby("variant")[["fit", "test_acc", "n_feats"]].mean().reindex(variants).round(4).to_string())
    P = A.pivot_table(index=["dataset", "seed"], columns="variant", values="fit")
    print(f"\n{main} against each variant (fitness, all runs pooled): W/T/L for {main}, two-sided sign test")
    for v in variants:
        if v == main or v not in P: continue
        a, b = P[main].dropna().align(P[v].dropna(), join="inner")
        w, t, l = wtl(a, b, False); p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
        print(f"  vs {v:<28s} {w:>3}/{t}/{l:<3}  p = {p:.4f}")

# ================= Extra helpers for the extension notebooks =================
def record(res, prob, t0, test_fn, cache, key_fn):
    m = np.asarray(res["mask"]).astype(int); sec = time.time() - t0
    return {"fit": float(res["fit"]), "cv": float(cache[key_fn(m)]["acc"]), "n_feats": int(m.sum()), "mask": np.packbits(m),
            "n": len(m), "hist": res["hist"], "arms": res.get("arms"), "calls": prob.calls, "sec": sec,
            "fsec": prob.fsec, "overhead_sec": sec - prob.fsec, "n_unique": len(prob.seen), **test_fn(m)}

def frame(store, cols=("fit", "cv", "n_feats", "test_acc", "test_prec", "test_rec", "test_f1")):
    return pd.DataFrame([{"dataset": k[0], "method": k[1], "seed": k[2], **{c: v.get(c, np.nan) for c in cols}} for k, v in store.items()])

def sign_line(R, a, b, label, metric="fit", higher_better=False):
    P = R.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
    if a not in P or b not in P: print(f"{label}: missing runs"); return None
    x, y = P[a].dropna().align(P[b].dropna(), join="inner")
    w, t, l = wtl(x, y, higher_better); p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
    print(f"{label}: {a} W/T/L = {w}/{t}/{l} over {len(x)} paired runs (mean diff {float((x - y).mean()):+.5f}), sign test p = {p:.4f}")
    return dict(label=label, a=a, b=b, W=w, T=t, L=l, n=len(x), p=p)
print("Extension helpers ready.")

# Pure algorithm overhead: every method on a fitness that costs (almost) nothing, so the time measured is
# the optimiser's own bookkeeping (position updates, tabu look-ups, controller). Same budgets and populations.
def overhead_benchmark(dims=(14, 44, 2000, 7129), seeds=(0, 1, 2), methods=None):
    methods = methods or ["RL-DOHO", "DO", "HO", "DOHO", "GA", "BPSO", "BGWO", "GA+SI", "HO+SI"]
    rows = []
    for D in dims:
        N, B = (12, 252) if D == 14 else (20, 1020)
        init = "sparse" if D > 100 else "dense"
        w = np.random.default_rng(123).normal(size=D)
        cheap = lambda m: 0.5 - 0.4 * np.tanh(float(w @ m) / 10) + 0.01 * m.sum() / D
        for a in methods:
            for s in seeds:
                prob = Problem(cheap, D, B); t0 = time.perf_counter()
                RUNNERS[a](prob, N, s, init=init)
                rows.append(dict(D=D, method=a, seed=s, total_sec=time.perf_counter() - t0, fit_sec=prob.fsec))
    T = pd.DataFrame(rows); T["overhead_ms_per_eval"] = 1000 * (T.total_sec - T.fit_sec) / np.where(T.D == 14, 252, 1020)
    return T.pivot_table(index="method", columns="D", values="overhead_ms_per_eval").reindex(methods)

Statistics helpers ready.
Extension helpers ready.


## 6. All runs (resumable)

Each job is (dataset, method or variant, seed). Results go to `v3_runs.pkl` (same budget) and `v3_long.pkl` (3× budget).
Variant names:
- `RL-DOHO [Q-learning]`, `RL-DOHO [random arm]`, `RL-DOHO [PERTURB only]`, `RL-DOHO [no intervention]`, `RL-DOHO [κ=1]` and so on.
- At 3× the budget: `UCB1 (3x)`, `Random arm (3x)`, `Q-learning (3x)`.

In [7]:
RES_FILE, LONG_FILE = f"{OUT_DIR}/v3_runs.pkl", f"{OUT_DIR}/v3_long.pkl"
runs  = pickle.load(open(RES_FILE, "rb")) if os.path.exists(RES_FILE) else {}
longr = pickle.load(open(LONG_FILE, "rb")) if os.path.exists(LONG_FILE) else {}
print(f"{len(runs)} same-budget runs and {len(longr)} long-budget runs already saved")

VARIANTS = {"RL-DOHO [Q-learning]": dict(controller="qlearn"), "RL-DOHO [random arm]": dict(controller="random"),
            "RL-DOHO [PERTURB only]": dict(controller="PERTURB"), "RL-DOHO [no intervention]": dict(intervene=False),
            "RL-DOHO [κ=1]": dict(kappa=1), "RL-DOHO [κ=3]": dict(kappa=3), "RL-DOHO [κ=4]": dict(kappa=4)}
LONG_VARIANTS = {"UCB1 (3x)": dict(), "Random arm (3x)": dict(controller="random"), "Q-learning (3x)": dict(controller="qlearn")}
REUSED_GENE = {"RL-DOHO [random arm]", "RL-DOHO [PERTURB only]", "RL-DOHO [no intervention]", "UCB1 (3x)", "Random arm (3x)"}

def jobs_for(ds):
    same = []
    if ds in TABS and RUN["tab_main"]: same += [(a, RUNNERS[a], {}) for a in WRAPPERS]
    if RUN["addons"]: same += [(a, RUNNERS[a], {}) for a in ADDONS]
    if RUN["variants"]: same += [(v, run_rldoho, kw) for v, kw in VARIANTS.items() if not (ds in GENE and v in REUSED_GENE)]
    long = [(v, run_rldoho, kw) for v, kw in LONG_VARIANTS.items() if RUN["long"] and not (ds in GENE and v in REUSED_GENE)]
    return same, long

t_start = time.time()
for ds in DATASETS:
    for s in SEEDS:
        same, long = jobs_for(ds)
        todo_f = [a for a in FILTERS if ds in TABS and RUN["tab_main"] and (ds, a, s) not in runs]
        same = [j for j in same if (ds, j[0], s) not in runs]; long = [j for j in long if (ds, j[0], s) not in longr]
        if not (same or long or todo_f): continue
        use_dataset(ds, s); init = init_for(N_FEATURES)
        for name, fn, kw in same:
            t0 = time.time(); prob = Problem(fitness_function, N_FEATURES, BUDGET)
            runs[(ds, name, s)] = record(fn(prob, POP_SIZE, s, init=init, verbose=VERBOSE, **kw), prob, t0, test_metrics, _cache, mask_str)
            pickle.dump(runs, open(RES_FILE, "wb"))
        for a in todo_f:
            t0 = time.time(); m, f, k = run_filter(a, s, X_tr, y_tr, fitness_function, N_FEATURES)
            runs[(ds, a, s)] = {"fit": float(f), "cv": float(_cache[mask_str(m)]["acc"]), "n_feats": int(np.sum(m)), "k": k,
                                "sec": time.time() - t0, **test_metrics(m)}
            pickle.dump(runs, open(RES_FILE, "wb"))
        for name, fn, kw in long:
            t0 = time.time(); prob = Problem(fitness_function, N_FEATURES, LONG_BUDGET)
            longr[(ds, name, s)] = record(fn(prob, POP_SIZE, s, init=init, verbose=VERBOSE, **kw), prob, t0, test_metrics, _cache, mask_str)
            pickle.dump(longr, open(LONG_FILE, "wb"))
        save_cache_now()
        show = [n for n in ["RL-DOHO", "GA", "HO", "GA+SI", "HO+SI", "RL-DOHO [Q-learning]"] if (ds, n, s) in runs]
        print(f"[{time.strftime('%H:%M:%S')}] {ds} seed {s}: " + "  ".join(f"{n}={runs[(ds, n, s)]['fit']:.4f}" for n in show)
              + f"   elapsed {time.time() - t_start:.0f}s", flush=True)
pickle.dump(NEW_EVAL, open(f"{OUT_DIR}/v3_eval_cost.pkl", "wb"))
print("All runs complete.")

587 same-budget runs and 92 long-budget runs already saved
[15:17:25] Hepatitis seed 4: RL-DOHO=0.2005  GA=0.2307  HO=0.2146  GA+SI=0.1819  HO+SI=0.2091  RL-DOHO [Q-learning]=0.2574   elapsed 125s
[15:20:12] Hepatitis seed 5: RL-DOHO=0.1962  GA=0.1937  HO=0.1866  GA+SI=0.1937  HO+SI=0.1861  RL-DOHO [Q-learning]=0.1909   elapsed 292s
[15:22:57] Hepatitis seed 6: RL-DOHO=0.2602  GA=0.2679  HO=0.2476  GA+SI=0.2413  HO+SI=0.2476  RL-DOHO [Q-learning]=0.2485   elapsed 457s
[15:25:40] Hepatitis seed 7: RL-DOHO=0.1578  GA=0.1320  HO=0.2269  GA+SI=0.2016  HO+SI=0.1797  RL-DOHO [Q-learning]=0.2330   elapsed 620s
[15:28:19] Hepatitis seed 8: RL-DOHO=0.2157  GA=0.2161  HO=0.2205  GA+SI=0.2115  HO+SI=0.2089  RL-DOHO [Q-learning]=0.2172   elapsed 779s
[15:31:13] Hepatitis seed 9: RL-DOHO=0.2061  GA=0.1697  HO=0.2041  GA+SI=0.1677  HO+SI=0.1879  RL-DOHO [Q-learning]=0.1700   elapsed 953s
[15:34:14] Dermatology seed 0: RL-DOHO=0.0336  GA=0.0203  HO=0.0395  GA+SI=0.0203  HO+SI=0.0406  RL-DOHO [Q-learn

## 7. Combine with the Notebook 7 results (gene data) into one table

In [8]:
cols = ["fit", "cv", "n_feats", "test_acc", "test_prec", "test_rec", "test_f1"]
def load(p): return pickle.load(open(p, "rb")) if os.path.exists(p) else {}
v2runs, v2abl, v2long, nb5 = load(f"{V2_DIR}/v2_runs.pkl"), load(f"{V2_DIR}/v2_ablation.pkl"), load(f"{V2_DIR}/v2_long.pkl"), load(NB5_FILE)
print(f"Notebook 7: {len(v2runs)} runs, {len(v2abl)} ablation runs, {len(v2long)} long runs | Notebook 5 filters: {len(nb5)}")
ABL_NAMES = {"Random arm": "RL-DOHO [random arm]", "PERTURB only": "RL-DOHO [PERTURB only]", "No intervention (DO only)": "RL-DOHO [no intervention]"}
LONG_NAMES = {"UCB, gain reward": "UCB1 (3x)", "Random arm": "Random arm (3x)"}
rows = []
for (ds, a, s), v in runs.items(): rows.append({"dataset": ds, "method": a, "seed": s, **{c: v[c] for c in cols}})
for (ds, a, s), v in v2runs.items():
    if ds in GENE: rows.append({"dataset": ds, "method": a, "seed": s, **{c: v[c] for c in cols}})
for (ds, a, s), v in v2abl.items():
    if ds in GENE and a in ABL_NAMES: rows.append({"dataset": ds, "method": ABL_NAMES[a], "seed": s, **{c: v[c] for c in cols}})
for (ds, a, s), v in nb5.items():
    if ds in GENE and a in FILTERS: rows.append({"dataset": ds, "method": a, "seed": s, **{c: v[c] for c in cols}})
R = pd.DataFrame(rows); R = R[R.dataset.isin(DATASETS) & R.seed.isin(SEEDS)].drop_duplicates(["dataset", "method", "seed"], keep="first")
lrows = [{"dataset": k[0], "method": k[1], "seed": k[2], **{c: v[c] for c in cols}} for k, v in longr.items()]
lrows += [{"dataset": k[0], "method": LONG_NAMES[k[1]], "seed": k[2], **{c: v[c] for c in cols}} for k, v in v2long.items()
          if k[0] in GENE and k[1] in LONG_NAMES]
L = pd.DataFrame(lrows); L = L[L.dataset.isin(DATASETS) & L.seed.isin(SEEDS)]
R.to_csv(f"{OUT_DIR}/v3_all_runs.csv", index=False); L.to_csv(f"{OUT_DIR}/v3_long_runs.csv", index=False)
print(R.groupby("method").size().to_string())

Notebook 7: 350 runs, 250 ablation runs, 150 long runs | Notebook 5 filters: 550
method
All features                 100
BGWO                         100
BPSO                         100
DO                           100
DOHO                         100
GA                           100
GA+SI                        100
HO                           100
HO+SI                        100
LASSO                        100
RL-DOHO                      100
RL-DOHO [PERTURB only]       100
RL-DOHO [Q-learning]         100
RL-DOHO [no intervention]    100
RL-DOHO [random arm]         100
RL-DOHO [κ=1]                100
RL-DOHO [κ=3]                100
RL-DOHO [κ=4]                100
ReliefF                      100
mRMR                         100


## 8. Main benchmark on the five new medical tabular datasets

In [9]:
pd.set_option("display.width", 220)
RT = R[R.dataset.isin(TABS) & R.method.isin(METHODS)]
S = summary_table(RT, METHODS)
for ds in TABS:
    T = S[S.dataset == ds].set_index("method").reindex(METHODS)
    rk = mean_ranks(RT[RT.dataset == ds], METHODS, "fit", False)[ds]; ra = mean_ranks(RT[RT.dataset == ds], METHODS, "test_acc", True)[ds]
    print(f"\n{'=' * 22} {ds} {'=' * 22}")
    print(T.assign(rank_fit=rk, rank_acc=ra)[["fit", "fit_sd", "cv", "test_acc", "acc_sd", "test_f1", "n_feats", "rank_fit", "rank_acc"]].round(4).to_string())
W = pd.concat([per_dataset_tests(RT, "RL-DOHO", METHODS, m, hb) for m, hb in [("fit", False), ("test_acc", True), ("test_f1", True)]])
POOL = pd.concat([pooled_sign(RT, "RL-DOHO", [m for m in METHODS if m != "RL-DOHO"], m, hb) for m, hb in [("fit", False), ("test_acc", True), ("test_f1", True)]])
print("\n=== New datasets pooled (50 paired runs): RL-DOHO W/T/L, two-sided sign test, Holm-corrected per metric ===")
print(POOL.round(4).to_string(index=False))
print("\nPer-dataset Holm-corrected Wilcoxon, fitness:")
print(W[W.metric == "fit"][["dataset", "vs", "W", "T", "L", "mean_diff", "holm_p", "friedman_p"]].round(4).to_string(index=False))
print("\nMean per-run rank on fitness (1 = best):"); print(mean_ranks(RT, METHODS, "fit", False).assign(ALL=lambda x: x.mean(1)).round(2).to_string())
print("\nMean per-run rank on test score:"); print(mean_ranks(RT, METHODS, "test_acc", True).assign(ALL=lambda x: x.mean(1)).round(2).to_string())
W.to_csv(f"{OUT_DIR}/v3_tab_stats_per_dataset.csv", index=False); POOL.to_csv(f"{OUT_DIR}/v3_tab_stats_pooled.csv", index=False)
S.to_csv(f"{OUT_DIR}/v3_tab_summary.csv", index=False)


====================== WDBC ======================
                 fit  fit_sd      cv  test_acc  acc_sd  test_f1  n_feats  rank_fit  rank_acc
method                                                                                      
RL-DOHO       0.0293  0.0050  0.9746    0.9435  0.0125   0.9493     12.7      4.20      7.35
DO            0.0409  0.0060  0.9640    0.9417  0.0157   0.9474     15.6      9.25      7.65
HO            0.0319  0.0048  0.9721    0.9446  0.0169   0.9500     12.9      5.70      5.95
DOHO          0.0336  0.0045  0.9706    0.9460  0.0148   0.9518     13.5      5.95      5.75
GA            0.0252  0.0053  0.9794    0.9474  0.0155   0.9525     14.3      1.45      5.95
BPSO          0.0289  0.0044  0.9758    0.9516  0.0135   0.9559     14.7      3.60      4.35
BGWO          0.0285  0.0048  0.9761    0.9447  0.0178   0.9494     14.3      3.15      6.30
LASSO         0.0333  0.0057  0.9702    0.9481  0.0110   0.9544     11.3      6.40      5.95
mRMR          0.03

## 9. Do the interventions help GA and HO? (H6, all ten datasets)

In [10]:
res6 = [sign_line(R, "GA+SI", "GA", "H6a  GA+SI vs GA (fitness)"), sign_line(R, "HO+SI", "HO", "H6b  HO+SI vs HO (fitness)")]
res6 = [r for r in res6 if r]
if res6:
    for r, h in zip(res6, holm([r["p"] for r in res6])): print(f"     Holm-corrected p for {r['a']} vs {r['b']}: {h:.4f}")
print()
for a, b in [("GA+SI", "GA"), ("HO+SI", "HO")]:
    sign_line(R, a, b, f"       test score {a} vs {b}", "test_acc", True)
print("\nOther comparisons (reported, no direction predicted):")
for a, b in [("RL-DOHO", "GA+SI"), ("RL-DOHO", "HO+SI"), ("GA+SI", "HO+SI")]:
    sign_line(R, a, b, f"  {a} vs {b}")
print("\nPer group of datasets:")
for grp, dss in [("gene", GENE), ("tabular", TABS)]:
    Rg = R[R.dataset.isin(dss)]
    for a, b in [("GA+SI", "GA"), ("HO+SI", "HO"), ("RL-DOHO", "GA+SI")]: sign_line(Rg, a, b, f"  [{grp}] {a} vs {b}")
names = ["RL-DOHO", "GA", "GA+SI", "HO", "HO+SI"]
print("\nMean fitness per dataset:"); print(R[R.method.isin(names)].pivot_table(index="method", columns="dataset", values="fit").reindex(names).round(4).to_string())
print("\nMean features selected per dataset:"); print(R[R.method.isin(names)].pivot_table(index="method", columns="dataset", values="n_feats").reindex(names).round(1).to_string())

H6a  GA+SI vs GA (fitness): GA+SI W/T/L = 43/11/46 over 100 paired runs (mean diff +0.00114), sign test p = 0.8323
H6b  HO+SI vs HO (fitness): HO+SI W/T/L = 70/6/24 over 100 paired runs (mean diff -0.00925), sign test p = 0.0000
     Holm-corrected p for GA+SI vs GA: 0.8323
     Holm-corrected p for HO+SI vs HO: 0.0000

       test score GA+SI vs GA: GA+SI W/T/L = 31/39/30 over 100 paired runs (mean diff -0.00031), sign test p = 1.0000
       test score HO+SI vs HO: HO+SI W/T/L = 34/32/34 over 100 paired runs (mean diff +0.00229), sign test p = 1.0000

Other comparisons (reported, no direction predicted):
  RL-DOHO vs GA+SI: RL-DOHO W/T/L = 23/0/77 over 100 paired runs (mean diff +0.01657), sign test p = 0.0000
  RL-DOHO vs HO+SI: RL-DOHO W/T/L = 41/1/58 over 100 paired runs (mean diff +0.00572), sign test p = 0.1074
  GA+SI vs HO+SI: GA+SI W/T/L = 70/0/30 over 100 paired runs (mean diff -0.01084), sign test p = 0.0001

Per group of datasets:
  [gene] GA+SI vs GA: GA+SI W/T/L = 25/0/25

## 10. Controllers: Q-learning vs UCB1 vs random arm (same budget, and H5 at 3× budget)

In [11]:
CTRL = ["RL-DOHO", "RL-DOHO [Q-learning]", "RL-DOHO [random arm]", "RL-DOHO [PERTURB only]", "RL-DOHO [no intervention]"]
A = R[R.method.isin(CTRL)].rename(columns={"method": "variant"})
ablation_report(A, "RL-DOHO", CTRL, "Controllers, same budget (RL-DOHO = UCB1)")
print()
sign_line(R, "RL-DOHO [Q-learning]", "RL-DOHO [random arm]", "Same budget: Q-learning vs random arm")
sign_line(R, "RL-DOHO [Q-learning]", "RL-DOHO", "Same budget: Q-learning vs UCB1")
if len(L):
    LL = L.rename(columns={"method": "variant"})
    ablation_report(LL, "Q-learning (3x)", list(LONG_VARIANTS), f"Controllers at budget {LONG_BUDGET}")
    print()
    h5 = sign_line(L, "Q-learning (3x)", "Random arm (3x)", "H5  Q-learning vs random arm at 3x budget")
    sign_line(L, "Q-learning (3x)", "UCB1 (3x)", "    Q-learning vs UCB1 at 3x budget")
    sign_line(L, "UCB1 (3x)", "Random arm (3x)", "    UCB1 vs random arm at 3x budget (H3 again, ten datasets)")
    for grp, dss in [("gene", GENE), ("tabular", TABS)]:
        sign_line(L[L.dataset.isin(dss)], "Q-learning (3x)", "Random arm (3x)", f"    [{grp}] Q-learning vs random arm at 3x")
# How many decisions does a controller actually get to make?
dec = pd.DataFrame([{"dataset": k[0], "method": k[1], "decisions": len(v["arms"]) if v.get("arms") is not None else 0}
                    for k, v in runs.items() if k[1] in ("RL-DOHO [Q-learning]", "GA+SI", "HO+SI")])
if len(dec): print("\nController decisions per run (mean):"); print(dec.pivot_table(index="method", columns="dataset", values="decisions").round(1).to_string())


=== Controllers, same budget (RL-DOHO = UCB1): mean per-run rank on fitness (1 = best) ===
dataset                    ALLAML  Backache  Dermatology  GLIOMA  GSE93272  Hepatitis  SPECTF  WDBC  colon  lung   ALL
variant                                                                                                               
RL-DOHO                      1.75       3.2          3.1    2.35       2.8       2.75     2.5  2.95   2.65  2.05  2.61
RL-DOHO [Q-learning]         2.80       2.9          3.0    3.00       3.3       3.15     2.8  1.95   2.45  2.75  2.81
RL-DOHO [random arm]         2.10       2.3          2.5    2.70       2.2       2.75     2.7  3.25   3.15  1.90  2.56
RL-DOHO [PERTURB only]       3.85       1.6          1.5    2.65       2.5       1.40     2.0  1.85   2.30  3.40  2.31
RL-DOHO [no intervention]    4.50       5.0          4.9    4.30       4.2       4.95     5.0  5.00   4.45  4.90  4.72

Mean fitness / test accuracy / features:
                              fit

## 11. Sensitivity to κ (H7)

In [12]:
from scipy.stats import friedmanchisquare
KAP = {"RL-DOHO [κ=1]": 1, "RL-DOHO": 2, "RL-DOHO [κ=3]": 3, "RL-DOHO [κ=4]": 4}
K = R[R.method.isin(KAP)].copy(); K["kappa"] = K.method.map(KAP)
P = K.pivot_table(index=["dataset", "seed"], columns="kappa", values="fit").dropna()
if P.shape[1] == 4 and len(P) > 1:
    print(f"H7  Friedman over kappa = 1..4 on {len(P)} paired runs: p = {friedmanchisquare(*[P[c] for c in P]).pvalue:.4f}")
    print("Mean per-run rank (1 = best):", P.rank(axis=1).mean().round(2).to_dict())
print("\nMean fitness per dataset:"); print(K.pivot_table(index="kappa", columns="dataset", values="fit").round(4).to_string())
print("\nMean test score per dataset:"); print(K.pivot_table(index="kappa", columns="dataset", values="test_acc").round(4).to_string())
for k in (1, 3, 4): sign_line(R, f"RL-DOHO [κ={k}]", "RL-DOHO", f"  kappa={k} vs kappa=2")

H7  Friedman over kappa = 1..4 on 100 paired runs: p = 0.0002
Mean per-run rank (1 = best): {1: 2.19, 2: 2.24, 3: 2.8, 4: 2.76}

Mean fitness per dataset:
dataset  ALLAML  Backache  Dermatology  GLIOMA  GSE93272  Hepatitis  SPECTF    WDBC   colon    lung
kappa                                                                                              
1        0.0654    0.2938       0.0220  0.1444    0.0744     0.1974  0.1700  0.0277  0.0759  0.0364
2        0.0516    0.3165       0.0247  0.1450    0.0791     0.2093  0.1671  0.0293  0.0831  0.0343
3        0.0834    0.3077       0.0243  0.1558    0.0797     0.2028  0.1791  0.0308  0.1020  0.0342
4        0.0655    0.3183       0.0251  0.1529    0.0904     0.2215  0.1662  0.0313  0.0974  0.0349

Mean test score per dataset:
dataset  ALLAML  Backache  Dermatology  GLIOMA  GSE93272  Hepatitis  SPECTF    WDBC   colon    lung
kappa                                                                                              
1        0.8045

## 12. Cost

- **Algorithm overhead** = wall-clock time minus the time spent inside the fitness function. Fitness results are cached and shared between methods on the same dataset and seed, so raw wall-clock depends on the order methods ran in; overhead does not.
- **Distinct subsets** = how many different subsets a run evaluated. With a fitness cache, this is what the run would cost from a cold start.
- **Estimated cold-start time** = overhead + distinct subsets × the mean cost of one new fitness evaluation on that dataset.

In [13]:
TM = ["RL-DOHO", "DO", "HO", "DOHO", "GA", "BPSO", "BGWO", "GA+SI", "HO+SI", "RL-DOHO [Q-learning]"]
rows = [{"dataset": k[0], "method": k[1], "seed": k[2], "overhead_sec": v["overhead_sec"], "n_unique": v["n_unique"]}
        for k, v in runs.items() if "overhead_sec" in v and k[1] in TM]
TT = pd.DataFrame(rows)
if len(TT):
    cost = {d: c[1] / max(c[0], 1) for d, c in NEW_EVAL.items()}
    print("Mean cost of one new fitness evaluation (s):", {d: round(c, 4) for d, c in cost.items()})
    TT["cold_sec"] = TT.overhead_sec + TT.n_unique * TT.dataset.map(cost).fillna(np.nan)
    for c, lab in [("overhead_sec", "Algorithm overhead per run (s)"), ("n_unique", "Distinct subsets evaluated per run"),
                   ("cold_sec", "Estimated cold-start time per run (s)")]:
        print(f"\n{lab}:"); print(TT.pivot_table(index="method", columns="dataset", values=c).reindex([m for m in TM if m in set(TT.method)]).round(2).to_string())
    TT.to_csv(f"{OUT_DIR}/v3_timing.csv", index=False)
if RUN["overhead"]:
    print("\nPure overhead on a near-free fitness (ms per fitness evaluation), by number of features D:")
    OB = overhead_benchmark(); print(OB.round(3).to_string()); OB.to_csv(f"{OUT_DIR}/v3_overhead_benchmark.csv")

Mean cost of one new fitness evaluation (s): {'Hepatitis': 0.0208, 'Dermatology': 0.0186, 'SPECTF': 0.0176, 'Backache': 0.0189}

Algorithm overhead per run (s):
dataset               ALLAML  Backache  Dermatology  GLIOMA  GSE93272  Hepatitis  SPECTF  WDBC  colon  lung
method                                                                                                     
RL-DOHO                  NaN      0.09         0.10     NaN       NaN       0.10    0.11  0.12    NaN   NaN
DO                       NaN      0.04         0.04     NaN       NaN       0.04    0.04  0.04    NaN   NaN
HO                       NaN      0.10         0.11     NaN       NaN       0.10    0.11  0.13    NaN   NaN
DOHO                     NaN      0.07         0.07     NaN       NaN       0.07    0.07  0.08    NaN   NaN
GA                       NaN      0.07         0.07     NaN       NaN       0.07    0.07  0.08    NaN   NaN
BPSO                     NaN      0.04         0.04     NaN       NaN       0.04   

## 13. Everything pooled: rank of every method and variant over the ten datasets

In [14]:
ALLM = METHODS + ADDONS + [v for v in VARIANTS]
RA = R[R.method.isin(ALLM)]
rk = mean_ranks(RA, [m for m in ALLM if m in set(RA.method)], "fit", False)
print(rk.assign(ALL=rk.mean(1)).sort_values("ALL").round(2).to_string())

dataset                    ALLAML  Backache  Dermatology  GLIOMA  GSE93272  Hepatitis  SPECTF   WDBC  colon   lung    ALL
method                                                                                                                   
GA+SI                        7.10      5.05         2.25    6.65      5.00       3.70    3.65   4.25   4.70   6.35   4.87
GA                           8.90      4.25         2.40    6.00      5.00       6.10    1.95   3.30   5.90   5.20   4.90
LASSO                        1.40     15.20        12.45    1.00      1.00      15.00   13.90  12.20   1.60   1.60   7.53
HO+SI                        9.40      7.15         7.05    6.25      7.95       6.75    6.10   8.30   6.20  10.60   7.58
RL-DOHO [PERTURB only]      13.50      3.60         5.80   10.10      9.15       4.25    6.10   4.60   7.20  13.55   7.78
RL-DOHO [κ=1]                9.50      7.00         7.30    8.40      9.90       5.50    9.10   5.75   7.50  11.90   8.18
RL-DOHO                 

## 14. Summary
Per-run results: `v3_runs.pkl`, `v3_long.pkl`, `v3_all_runs.csv`, `v3_long_runs.csv`. Statistics: `v3_tab_stats_*.csv`. Timing: `v3_timing.csv`, `v3_overhead_benchmark.csv`. All files are in `OUT_DIR`.